# Notebook 6 — SQL Analysis

## Purpose

This notebook demonstrates the use of SQL to analyze the Kenya Retail Sales dataset and answer practical business questions.

The analysis builds on the cleaned dataset prepared in Notebook 2 and the business questions identified in Notebooks 3 and 4.

Rather than demonstrating SQL syntax in isolation, the queries are designed to solve real retail business problems involving:

- Sales and profitability
- Branch and city performance
- Product performance
- Customer behavior
- Payment methods
- Employee performance
- Time-based sales patterns
- Ranking and performance comparison

The SQL analysis will progress from basic queries to advanced techniques such as:

- Aggregations
- CASE statements
- HAVING
- Subqueries
- Common Table Expressions (CTEs)
- Window functions
- Ranking
- Running totals
- Percentage calculations

In [1]:
# Import required libraries
import sqlite3
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/Kenya_Retail_Sales_Cleaned.csv"
)

# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Display dataset dimensions
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 50,000
Columns: 29


## 2. Create SQL Database

The cleaned Pandas DataFrame will be loaded into a SQLite database table called `retail_sales`.

This table will serve as the main source for the SQL queries throughout the notebook.

In [2]:
# Create SQLite database connection
conn = sqlite3.connect("../data/retail_sales.db")

# Load the cleaned dataset into SQLite
df.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully.")
print("Table 'retail_sales' created successfully.")

SQLite database created successfully.
Table 'retail_sales' created successfully.


## 3. Verify the SQL Table

Before beginning the analysis, the database structure and record count are verified to ensure that the SQL table contains the expected data.

In [3]:
# Check number of records
record_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_records
    FROM retail_sales;
    """,
    conn
)

record_count

,total_records
0,50000


In [4]:
# Inspect the SQL table structure
table_info = pd.read_sql_query(
    """
    PRAGMA table_info(retail_sales);
    """,
    conn
)

table_info

,cid,name,type,notnull,dflt_value,pk
0,0,Invoice_ID,TEXT,0,None,0
1,1,Date,TIMESTAMP,0,None,0
2,2,Time,TEXT,0,None,0
3,3,Branch,TEXT,0,None,0
4,4,City,TEXT,0,None,0
5,5,Customer_ID,TEXT,0,None,0
6,6,Customer_Type,TEXT,0,None,0
7,7,Gender,TEXT,0,None,0
8,8,Product_Category,TEXT,0,None,0
9,9,Product,TEXT,0,None,0


In [5]:
def run_query(query):
    """
    Execute a SQL query and return the result as a Pandas DataFrame.
    """
    return pd.read_sql_query(query, conn)

## 4. Basic SQL Queries

### Business Question

What does the retail sales table contain?

We begin by retrieving a small sample of records from the database to confirm that the data can be accessed successfully using SQL.

In [6]:
query = """
SELECT *
FROM retail_sales
LIMIT 10;
"""

run_query(query)

,Invoice_ID,Date,Time,Branch,City,Customer_ID,Customer_Type,Gender,Product_Category,Product,...,Cashier,Salesperson,Year,Month,Month Name,Quarter,Day,Day Name,Hour,Day Type
0,INV100000,2026-07-01 00:00:00,1900-01-01 16:41:00,Eldoret,Uasin Gishu,CUST3086,Member,Female,Electronics,Power Bank,...,Wanjiku,Grace,2026,7,July,3,1,Wednesday,16,Weekday
1,INV100001,2025-10-11 00:00:00,1900-01-01 15:11:00,Mombasa,Mombasa,CUST2099,Member,Male,Beverages,Tea,...,Mutua,Mercy,2025,10,October,4,11,Saturday,15,Weekend
2,INV100002,2025-04-24 00:00:00,1900-01-01 13:01:00,Kisumu,Kisumu,CUST7763,Member,Male,Groceries,Cooking Oil,...,Mutua,Brian,2025,4,April,2,24,Thursday,13,Weekday
3,INV100003,2026-02-13 00:00:00,1900-01-01 11:37:00,Kisumu,Kisumu,CUST1881,Normal,Female,Electronics,Power Bank,...,Wanjiku,John,2026,2,February,1,13,Friday,11,Weekday
4,INV100004,2026-08-30 00:00:00,1900-01-01 13:54:00,Nakuru,Nakuru,CUST8901,Normal,Male,Groceries,Milk,...,Wanjiku,Brian,2026,8,August,3,30,Sunday,13,Weekend
5,INV100005,2026-12-29 00:00:00,1900-01-01 10:01:00,Westlands,Nairobi,CUST6787,Member,Male,Beverages,Water,...,Achieng,John,2026,12,December,4,29,Tuesday,10,Weekday
6,INV100006,2025-09-05 00:00:00,1900-01-01 12:16:00,Eldoret,Uasin Gishu,CUST1166,Normal,Male,Beverages,Soda,...,Kamau,Grace,2025,9,September,3,5,Friday,12,Weekday
7,INV100007,2026-08-13 00:00:00,1900-01-01 12:34:00,Nakuru,Nakuru,CUST8300,Member,Male,Beverages,Water,...,Wanjiku,John,2026,8,August,3,13,Thursday,12,Weekday
8,INV100008,2026-10-29 00:00:00,1900-01-01 19:29:00,Migori,Migori,CUST6769,Member,Male,Stationery,Pen,...,Wanjiku,John,2026,10,October,4,29,Thursday,19,Weekday
9,INV100009,2026-01-12 00:00:00,1900-01-01 15:34:00,Nakuru,Nakuru,CUST9906,Member,Male,Stationery,File,...,Wanjiku,Grace,2026,1,January,1,12,Monday,15,Weekday


In [7]:
query = """
SELECT
    Invoice_ID,
    Date,
    Branch,
    City,
    Product,
    Quantity,
    Total_Sales,
    Profit
FROM retail_sales
LIMIT 10;
"""

run_query(query)

,Invoice_ID,Date,Branch,City,Product,Quantity,Total_Sales,Profit
0,INV100000,2026-07-01 00:00:00,Eldoret,Uasin Gishu,Power Bank,10,8035.70,2476.71
1,INV100001,2025-10-11 00:00:00,Mombasa,Mombasa,Tea,2,9876.73,3335.01
2,INV100002,2025-04-24 00:00:00,Kisumu,Kisumu,Cooking Oil,4,21918.34,7096.75
3,INV100003,2026-02-13 00:00:00,Kisumu,Kisumu,Power Bank,6,19635.20,7296.11
4,INV100004,2026-08-30 00:00:00,Nakuru,Nakuru,Milk,10,18649.80,3391.85
5,INV100005,2026-12-29 00:00:00,Westlands,Nairobi,Water,9,25002.13,8582.94
6,INV100006,2025-09-05 00:00:00,Eldoret,Uasin Gishu,Soda,7,17127.79,2672.49
7,INV100007,2026-08-13 00:00:00,Nakuru,Nakuru,Water,8,4678.95,1222.07
8,INV100008,2026-10-29 00:00:00,Migori,Migori,Pen,4,3370.90,916.49
9,INV100009,2026-01-12 00:00:00,Nakuru,Nakuru,File,5,19666.51,4589.10


## 5. DISTINCT Values

### Business Question

Which branches, cities, product categories, and payment methods are represented in the dataset?

Identifying unique categorical values helps analysts understand the structure of the business.

In [8]:
query = """
SELECT DISTINCT Branch
FROM retail_sales
ORDER BY Branch;
"""

run_query(query)

,Branch
0,Eldoret
1,Kisumu
2,Migori
3,Mombasa
4,Nairobi CBD
5,Nakuru
6,Westlands


In [9]:
query = """
SELECT DISTINCT Payment_Method
FROM retail_sales
ORDER BY Payment_Method;
"""

run_query(query)

,Payment_Method
0,Cash
1,Credit Card
2,Debit Card
3,M-Pesa


In [11]:
query = """
SELECT DISTINCT Product_Category
fROM retail_sales
ORDER BY Product_Category;
"""
run_query(query)    

,Product_Category
0,Beverages
1,Electronics
2,Groceries
3,Household
4,Stationery


## 6. Filtering Data with WHERE

### Business Question

Which transactions generated more than KES 50,000 in sales?

Filtering allows analysts to investigate specific segments of the transaction data.

In [13]:
query = """
SELECT
    Invoice_ID,
    Date,
    Branch,
    City,
    Product,
    Quantity,
    Total_Sales,
    Profit
FROM retail_sales
WHERE Total_Sales > 50000
ORDER BY Total_Sales DESC
LIMIT 20;
"""
run_query(query)

,Invoice_ID,Date,Branch,City,Product,Quantity,Total_Sales,Profit
0,INV145198,2026-05-14 00:00:00,Westlands,Nairobi,Book,10,57981.44,15896.98
1,INV119658,2025-03-17 00:00:00,Eldoret,Uasin Gishu,USB Flash,10,57908.94,15953.61
2,INV142385,2026-06-10 00:00:00,Eldoret,Uasin Gishu,Bucket,10,57894.44,18406.42
3,INV108452,2025-05-02 00:00:00,Kisumu,Kisumu,Juice,10,57807.09,10678.24
4,INV149110,2026-06-23 00:00:00,Nairobi CBD,Nairobi,Detergent,10,57799.78,16401.21
5,INV130812,2025-08-01 00:00:00,Nakuru,Nakuru,Water,10,57693.88,17353.83
6,INV146842,2026-03-02 00:00:00,Kisumu,Kisumu,Marker,10,57635.88,8933.06
7,INV144348,2026-06-28 00:00:00,Nairobi CBD,Nairobi,Headphones,10,57634.48,12892.11
8,INV113556,2026-10-05 00:00:00,Eldoret,Uasin Gishu,Keyboard,10,57586.34,18247.09
9,INV125676,2026-12-04 00:00:00,Eldoret,Uasin Gishu,Sugar,10,57547.83,14910.22


## 7. Sorting Business Results

### Business Question

Which individual transactions generated the highest profit?

Sorting allows management to quickly identify the largest contributors to profitability.

In [14]:
query = """
SELECT
    Invoice_ID,
    Date,
    Branch,
    Product,
    Total_Sales,
    Cost,
    Profit
FROM retail_sales
ORDER BY Profit DESC
LIMIT 10;
"""

run_query(query)

,Invoice_ID,Date,Branch,Product,Total_Sales,Cost,Profit
0,INV107078,2025-04-17 00:00:00,Westlands,Book,55351.49,26382.70,21334.10
1,INV104344,2025-12-03 00:00:00,Westlands,Tea,55312.86,26398.32,21285.18
2,INV116306,2025-06-06 00:00:00,Eldoret,Tea,54639.36,26338.80,20764.10
3,INV131645,2025-02-07 00:00:00,Nakuru,Rice,54616.28,26333.39,20749.61
4,INV133809,2026-11-24 00:00:00,Migori,Keyboard,56008.40,27657.98,20625.12
5,INV128665,2025-05-26 00:00:00,Mombasa,Pen,54134.65,26057.43,20610.37
6,INV148293,2025-03-03 00:00:00,Eldoret,Power Bank,53061.30,25282.67,20459.83
7,INV142108,2026-02-01 00:00:00,Migori,Rice,55964.20,27800.60,20444.40
8,INV125378,2026-03-29 00:00:00,Westlands,Milk,54341.13,26410.21,20435.59
9,INV115552,2025-02-01 00:00:00,Kisumu,Detergent,55936.24,27787.96,20432.94


## 8. Aggregating Sales by Branch

### Business Question

How much revenue does each branch generate?

Grouping transactions by branch allows management to compare sales performance across locations.

In [15]:
query = """
SELECT
    Branch,
    SUM(Total_Sales) AS Total_Revenue
FROM retail_sales
GROUP BY Branch
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Branch,Total_Revenue
0,Nakuru,1.069901e+08
1,Mombasa,1.069765e+08
2,Eldoret,1.064426e+08
3,Westlands,1.057603e+08
4,Migori,1.051070e+08
5,Kisumu,1.050057e+08
6,Nairobi CBD,1.047641e+08


### Business Question

Which branches generate the highest total profit?

Revenue alone does not show the complete financial performance of a branch, so profitability should also be evaluated.

In [22]:
query = """
SELECT
    Branch,
    SUM(Profit) AS Total_Profit
FROM retail_sales
GROUP BY Branch
ORDER BY Total_Profit DESC;
"""

run_query(query)

,Branch,Total_Profit
0,Eldoret,27584451.30
1,Mombasa,27537270.80
2,Nakuru,27530326.44
3,Migori,27311467.44
4,Westlands,27290291.29
5,Kisumu,27265106.86
6,Nairobi CBD,26842019.97


In [23]:
query = """
SELECT
    Branch,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    AVG(Total_Sales) AS Average_Transaction_Value
FROM retail_sales
GROUP BY Branch
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Branch,Transactions,Total_Revenue,Total_Profit,Average_Transaction_Value
0,Nakuru,7196,1.069901e+08,27530326.44,14868.001908
1,Mombasa,7272,1.069765e+08,27537270.80,14710.740513
2,Eldoret,7200,1.064426e+08,27584451.30,14783.691743
3,Westlands,7015,1.057603e+08,27290291.29,15076.313766
4,Migori,7012,1.051070e+08,27311467.44,14989.582289
5,Kisumu,7192,1.050057e+08,27265106.86,14600.352266
6,Nairobi CBD,7113,1.047641e+08,26842019.97,14728.532718


In [24]:
query = """
SELECT
    City,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    AVG(Total_Sales) AS Average_Transaction_Value
FROM retail_sales
GROUP BY City
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,City,Transactions,Total_Revenue,Total_Profit,Average_Transaction_Value
0,Nairobi,14128,2.105244e+08,54132311.26,14901.217036
1,Nakuru,7196,1.069901e+08,27530326.44,14868.001908
2,Mombasa,7272,1.069765e+08,27537270.80,14710.740513
3,Uasin Gishu,7200,1.064426e+08,27584451.30,14783.691743
4,Migori,7012,1.051070e+08,27311467.44,14989.582289
5,Kisumu,7192,1.050057e+08,27265106.86,14600.352266


## 10. Product Category Analysis

### Business Question

Which product categories generate the most revenue and profit?

Category-level analysis helps management understand which areas of the product portfolio contribute most to financial performance.

In [25]:
query = """
SELECT
    Product_Category,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    SUM(Quantity) AS Units_Sold,
    COUNT(DISTINCT Invoice_ID) AS Transactions
FROM retail_sales
GROUP BY Product_Category
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Product_Category,Total_Revenue,Total_Profit,Units_Sold,Transactions
0,Household,1.498403e+08,38468709.21,55756,10139
1,Beverages,1.494720e+08,38492597.87,55091,10058
2,Electronics,1.489817e+08,38594506.08,55022,9969
3,Stationery,1.467696e+08,37931377.66,54661,9956
4,Groceries,1.459827e+08,37873743.28,53994,9878


## 11. Customer Segment Analysis

### Business Question

How do different customer types contribute to revenue, profit, and transaction volume?

Understanding customer segments can help management evaluate customer value and develop appropriate retention and marketing strategies.

In [26]:
query = """
SELECT
    Customer_Type,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    AVG(Total_Sales) AS Average_Transaction_Value,
    AVG(Quantity) AS Average_Quantity
FROM retail_sales
GROUP BY Customer_Type
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Customer_Type,Transactions,Total_Revenue,Total_Profit,Average_Transaction_Value,Average_Quantity
0,Member,25137,3.727715e+08,96046786.68,14829.595508,5.502009
1,Normal,24863,3.682748e+08,95314147.42,14812.161195,5.478824


## 12. Payment Method Performance

### Business Question

How does each payment method contribute to transaction volume, revenue, and profit?

In [27]:
query = """
SELECT
    Payment_Method,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit
FROM retail_sales
GROUP BY Payment_Method
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Payment_Method,Transactions,Total_Revenue,Total_Profit
0,Cash,12562,1.872907e+08,48405339.19
1,Credit Card,12579,1.865366e+08,48278238.52
2,M-Pesa,12494,1.850064e+08,47611320.30
3,Debit Card,12365,1.822126e+08,47066036.09


## 13. Salesperson Performance

### Business Question

How much revenue and profit is generated by each salesperson?

Salesperson performance can be evaluated using revenue, profit, and transaction volume.

In [29]:
query = """
SELECT
    Salesperson,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    AVG(Total_Sales) AS Average_Transaction_Value
FROM retail_sales
GROUP BY Salesperson
ORDER BY Total_Revenue DESC;
"""

run_query(query)

,Salesperson,Transactions,Total_Revenue,Total_Profit,Average_Transaction_Value
0,Brian,8362,1.253229e+08,32232954.52,14987.191172
1,John,8409,1.248499e+08,32083434.37,14847.170383
2,Grace,8413,1.247396e+08,32266232.79,14827.010560
3,Mercy,8277,1.227443e+08,31927399.57,14829.566412
4,Faith,8346,1.225242e+08,31484092.26,14680.584514
5,Kevin,8193,1.208654e+08,31366820.59,14752.281018


## 14. Transaction Value Statistics

### Business Question

What are the minimum, maximum, and average transaction values in the dataset?

These statistics provide a high-level view of transaction size.

In [30]:
query = """
SELECT
    MIN(Total_Sales) AS Minimum_Sale,
    MAX(Total_Sales) AS Maximum_Sale,
    AVG(Total_Sales) AS Average_Sale,
    MIN(Profit) AS Minimum_Profit,
    MAX(Profit) AS Maximum_Profit,
    AVG(Profit) AS Average_Profit
FROM retail_sales;
"""

run_query(query)

,Minimum_Sale,Maximum_Sale,Average_Sale,Minimum_Profit,Maximum_Profit,Average_Profit
0,50.25,57981.44,14820.926122,10.67,21334.1,3827.218682


## 15. Overall Business KPIs

### Business Question

What is the overall financial and operational performance of the retail business?

The following SQL query calculates the key business metrics used throughout the project.

In [32]:
query = """
SELECT
    COUNT(DISTINCT Invoice_ID) AS Total_Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    AVG(Total_Sales) AS Average_Transaction_Value,
    AVG(Profit) AS Average_Profit_Per_Transaction
FROM retail_sales;
"""

run_query(query)

,Total_Transactions,Total_Revenue,Total_Profit,Average_Transaction_Value,Average_Profit_Per_Transaction
0,50000,7.410463e+08,191360934.1,14820.926122,3827.218682


In [33]:
query = """
SELECT
    Branch,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    ROUND(
        SUM(Profit) * 100.0 / NULLIF(SUM(Total_Sales), 0),
        2
    ) AS Profit_Margin_Percent
FROM retail_sales
GROUP BY Branch
ORDER BY Profit_Margin_Percent DESC;
"""

run_query(query)

,Branch,Total_Revenue,Total_Profit,Profit_Margin_Percent
0,Migori,1.051070e+08,27311467.44,25.98
1,Kisumu,1.050057e+08,27265106.86,25.97
2,Eldoret,1.064426e+08,27584451.30,25.91
3,Westlands,1.057603e+08,27290291.29,25.80
4,Mombasa,1.069765e+08,27537270.80,25.74
5,Nakuru,1.069901e+08,27530326.44,25.73
6,Nairobi CBD,1.047641e+08,26842019.97,25.62


In [34]:
# ================================================================
# NOTEBOOK 6 — ADVANCED SQL BUSINESS ANALYSIS
# Kenya Retail Sales Analysis & Business Intelligence Dashboard
# ================================================================
#
# This section demonstrates advanced SQL techniques using SQLite.
#
# Techniques covered:
# 1. HAVING
# 2. CASE statements
# 3. Subqueries
# 4. Common Table Expressions (CTEs)
# 5. JOINs
# 6. Window functions
# 7. Ranking
# 8. Running totals
# 9. Percentage-of-total calculations
#
# IMPORTANT:
# All analysis is performed using the cleaned dataset already
# loaded into the SQLite table called "retail_sales".
# ================================================================


# ================================================================
# 1. HAVING
# ================================================================
#
# WHERE filters individual rows BEFORE aggregation.
#
# HAVING filters GROUPED results AFTER aggregation.
#
# Business Question:
# Which branches generated more than KES 200 million in revenue?
#
# We first GROUP BY Branch, calculate total revenue for each branch,
# and then use HAVING to keep only branches above the threshold.
# ================================================================

query_having = """
SELECT
    Branch,
    COUNT(DISTINCT Invoice_ID) AS Transactions,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit
FROM retail_sales
GROUP BY Branch
HAVING SUM(Total_Sales) > 200000000
ORDER BY Total_Revenue DESC;
"""

having_result = run_query(query_having)

print("Branches generating more than KES 200 million in revenue:")
display(having_result)


# ================================================================
# 2. HAVING — PRODUCT CATEGORY
# ================================================================
#
# Business Question:
# Which product categories generated more than KES 30 million
# in profit?
#
# Again, HAVING is used because we are filtering an aggregated
# value (SUM of Profit), not an individual transaction.
# ================================================================

query_category_having = """
SELECT
    Product_Category,
    SUM(Total_Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    SUM(Quantity) AS Units_Sold
FROM retail_sales
GROUP BY Product_Category
HAVING SUM(Profit) > 30000000
ORDER BY Total_Profit DESC;
"""

category_having_result = run_query(query_category_having)

print("Product categories generating more than KES 30 million profit:")
display(category_having_result)


# ================================================================
# 3. CASE STATEMENT — CLASSIFY TRANSACTIONS
# ================================================================
#
# CASE allows SQL to create business categories based on conditions.
#
# Business Question:
# How can we classify transactions according to their sales value?
#
# We create three categories:
#
# High Value     -> Sales >= KES 30,000
# Medium Value   -> Sales between KES 10,000 and KES 29,999
# Low Value      -> Sales below KES 10,000
#
# This demonstrates how SQL can transform raw numerical values
# into meaningful business categories.
# ================================================================

query_case = """
SELECT
    Invoice_ID,
    Total_Sales,
    CASE
        WHEN Total_Sales >= 30000 THEN 'High Value'
        WHEN Total_Sales >= 10000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS Transaction_Value_Category
FROM retail_sales
LIMIT 20;
"""

case_result = run_query(query_case)

print("Example transaction value classifications:")
display(case_result)


# ================================================================
# 4. CASE STATEMENT — PROFITABILITY CLASSIFICATION
# ================================================================
#
# Business Question:
# How can transactions be classified according to profitability?
#
# A CASE statement allows us to create a simple profitability
# classification based on the Profit column.
# ================================================================

query_profit_case = """
SELECT
    Invoice_ID,
    Total_Sales,
    Profit,
    CASE
        WHEN Profit > 10000 THEN 'High Profit'
        WHEN Profit > 5000 THEN 'Medium Profit'
        ELSE 'Low Profit'
    END AS Profit_Category
FROM retail_sales
LIMIT 20;
"""

profit_case_result = run_query(query_profit_case)

print("Example profitability classifications:")
display(profit_case_result)


# ================================================================
# 5. CASE + GROUP BY
# ================================================================
#
# CASE becomes even more useful when combined with GROUP BY.
#
# Business Question:
# How many transactions fall into each sales-value category?
#
# COUNT(*) counts the transactions in each category.
# ================================================================

query_case_summary = """
SELECT
    CASE
        WHEN Total_Sales >= 30000 THEN 'High Value'
        WHEN Total_Sales >= 10000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS Transaction_Value_Category,

    COUNT(*) AS Transaction_Count,

    SUM(Total_Sales) AS Total_Revenue,

    SUM(Profit) AS Total_Profit

FROM retail_sales

GROUP BY
    CASE
        WHEN Total_Sales >= 30000 THEN 'High Value'
        WHEN Total_Sales >= 10000 THEN 'Medium Value'
        ELSE 'Low Value'
    END

ORDER BY Total_Revenue DESC;
"""

case_summary_result = run_query(query_case_summary)

print("Transaction value category summary:")
display(case_summary_result)


# ================================================================
# 6. SUBQUERY — ABOVE-AVERAGE TRANSACTIONS
# ================================================================
#
# A subquery is a query inside another query.
#
# Business Question:
# Which transactions generated sales above the overall average
# transaction value?
#
# The inner query calculates the average Total_Sales.
#
# The outer query then finds transactions greater than that
# calculated average.
#
# This is a useful example because the threshold is calculated
# dynamically from the dataset rather than being manually entered.
# ================================================================

query_subquery = """
SELECT
    Invoice_ID,
    Date,
    Branch,
    Product,
    Total_Sales,
    Profit
FROM retail_sales
WHERE Total_Sales > (
    SELECT AVG(Total_Sales)
    FROM retail_sales
)
ORDER BY Total_Sales DESC
LIMIT 20;
"""

subquery_result = run_query(query_subquery)

print("Transactions above the overall average transaction value:")
display(subquery_result)


# ================================================================
# 7. SUBQUERY — ABOVE-AVERAGE PRODUCT REVENUE
# ================================================================
#
# Business Question:
# Which products generate more revenue than the average product
# revenue?
#
# The inner query calculates revenue for every product.
#
# The outer query calculates the average of those product revenues
# and returns products above that average.
#
# This demonstrates nested aggregation.
# ================================================================

query_product_subquery = """
SELECT
    Product,
    SUM(Total_Sales) AS Total_Revenue
FROM retail_sales
GROUP BY Product
HAVING SUM(Total_Sales) > (
    SELECT AVG(Product_Revenue)
    FROM (
        SELECT
            Product,
            SUM(Total_Sales) AS Product_Revenue
        FROM retail_sales
        GROUP BY Product
    )
)
ORDER BY Total_Revenue DESC;
"""

product_subquery_result = run_query(query_product_subquery)

print("Products generating above-average product revenue:")
display(product_subquery_result)


# ================================================================
# 8. COMMON TABLE EXPRESSION (CTE)
# ================================================================
#
# A CTE allows us to create a temporary named result that can then
# be used by the main query.
#
# CTEs make complicated SQL easier to read and maintain.
#
# Business Question:
# Which branches have a profit margin above 25%?
#
# Step 1:
# Calculate revenue and profit for each branch.
#
# Step 2:
# Calculate profit margin.
#
# Step 3:
# Filter branches using the calculated margin.
# ================================================================

query_cte = """
WITH branch_performance AS (

    SELECT
        Branch,
        SUM(Total_Sales) AS Total_Revenue,
        SUM(Profit) AS Total_Profit

    FROM retail_sales

    GROUP BY Branch
)

SELECT
    Branch,
    Total_Revenue,
    Total_Profit,

    ROUND(
        Total_Profit * 100.0 /
        NULLIF(Total_Revenue, 0),
        2
    ) AS Profit_Margin_Percent

FROM branch_performance

WHERE
    Total_Profit * 100.0 /
    NULLIF(Total_Revenue, 0) > 25

ORDER BY Profit_Margin_Percent DESC;
"""

cte_result = run_query(query_cte)

print("Branches with profit margins above 25%:")
display(cte_result)


# ================================================================
# 9. JOIN — BRANCH PERFORMANCE + CITY INFORMATION
# ================================================================
#
# JOINs combine information from different tables.
#
# Our original retail_sales table contains both Branch and City,
# but for demonstration purposes we can create a separate lookup
# table containing each branch and its city.
#
# This demonstrates a fundamental relational database concept:
#
# Fact table:
#     retail_sales
#
# Dimension / lookup table:
#     branch_city
# ================================================================

# Create a branch-city lookup table from the existing data.
# DISTINCT ensures that each branch-city combination appears once.

branch_city = df[["Branch", "City"]].drop_duplicates()

branch_city.to_sql(
    "branch_city",
    conn,
    if_exists="replace",
    index=False
)


# Now join the sales table to the branch lookup table.

query_join = """
SELECT
    r.Branch,
    b.City,
    COUNT(DISTINCT r.Invoice_ID) AS Transactions,
    SUM(r.Total_Sales) AS Total_Revenue,
    SUM(r.Profit) AS Total_Profit

FROM retail_sales AS r

INNER JOIN branch_city AS b
    ON r.Branch = b.Branch

GROUP BY
    r.Branch,
    b.City

ORDER BY
    Total_Revenue DESC;
"""

join_result = run_query(query_join)

print("Branch performance joined with city information:")
display(join_result)


# ================================================================
# 10. WINDOW FUNCTION — RANK BRANCHES
# ================================================================
#
# Window functions perform calculations across related rows
# without collapsing those rows into a single GROUP BY result.
#
# Business Question:
# What is the revenue ranking of each branch?
#
# First we calculate branch revenue.
#
# Then RANK() assigns a position to each branch.
#
# Unlike GROUP BY alone, the ranking information is retained
# alongside each branch's revenue.
# ================================================================

query_rank = """
WITH branch_revenue AS (

    SELECT
        Branch,
        SUM(Total_Sales) AS Total_Revenue

    FROM retail_sales

    GROUP BY Branch
)

SELECT
    Branch,
    Total_Revenue,

    RANK() OVER (
        ORDER BY Total_Revenue DESC
    ) AS Revenue_Rank

FROM branch_revenue

ORDER BY Revenue_Rank;
"""

rank_result = run_query(query_rank)

print("Branch revenue ranking:")
display(rank_result)


# ================================================================
# 11. WINDOW FUNCTION — RANK SALESPERSONS
# ================================================================
#
# Business Question:
# How do salespeople rank according to total revenue?
#
# This is another example of a window function.
#
# Important:
# RANK() does not remove the individual salesperson rows.
# It simply adds a ranking value to each row.
# ================================================================

query_salesperson_rank = """
WITH salesperson_performance AS (

    SELECT
        Salesperson,
        COUNT(DISTINCT Invoice_ID) AS Transactions,
        SUM(Total_Sales) AS Total_Revenue,
        SUM(Profit) AS Total_Profit

    FROM retail_sales

    GROUP BY Salesperson
)

SELECT
    Salesperson,
    Transactions,
    Total_Revenue,
    Total_Profit,

    RANK() OVER (
        ORDER BY Total_Revenue DESC
    ) AS Revenue_Rank

FROM salesperson_performance

ORDER BY Revenue_Rank;
"""

salesperson_rank_result = run_query(query_salesperson_rank)

print("Salesperson revenue ranking:")
display(salesperson_rank_result)


# ================================================================
# 12. WINDOW FUNCTION — RUNNING TOTAL
# ================================================================
#
# Running totals are useful for understanding how performance
# accumulates over time.
#
# Business Question:
# How does cumulative revenue build throughout the year?
#
# Step 1:
# Aggregate revenue by month.
#
# Step 2:
# Use SUM() as a window function to calculate cumulative revenue.
#
# ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW means:
#
# Start from the first available month
# and continue through the current month.
# ================================================================

query_running_total = """
WITH monthly_sales AS (

    SELECT
        Year,
        Month,
        SUM(Total_Sales) AS Monthly_Revenue

    FROM retail_sales

    GROUP BY
        Year,
        Month
)

SELECT
    Year,
    Month,
    Monthly_Revenue,

    SUM(Monthly_Revenue) OVER (
        ORDER BY Year, Month
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS Cumulative_Revenue

FROM monthly_sales

ORDER BY
    Year,
    Month;
"""

running_total_result = run_query(query_running_total)

print("Monthly revenue and cumulative revenue:")
display(running_total_result)


# ================================================================
# 13. PERCENTAGE OF TOTAL REVENUE
# ================================================================
#
# Business Question:
# What percentage of the company's total revenue does each branch
# contribute?
#
# Step 1:
# Calculate revenue for each branch.
#
# Step 2:
# Calculate total company revenue.
#
# Step 3:
# Divide branch revenue by total revenue.
#
# Multiplying by 100 converts the result into a percentage.
# ================================================================

query_percentage = """
WITH branch_revenue AS (

    SELECT
        Branch,
        SUM(Total_Sales) AS Branch_Revenue

    FROM retail_sales

    GROUP BY Branch
)

SELECT
    Branch,
    Branch_Revenue,

    ROUND(
        Branch_Revenue * 100.0 /
        SUM(Branch_Revenue) OVER (),
        2
    ) AS Revenue_Percentage

FROM branch_revenue

ORDER BY Branch_Revenue DESC;
"""

percentage_result = run_query(query_percentage)

print("Branch contribution to total company revenue:")
display(percentage_result)


# ================================================================
# 14. PROFIT MARGIN BY PRODUCT CATEGORY
# ================================================================
#
# Business Question:
# Which product categories generate the strongest profit margins?
#
# Revenue alone does not tell us how efficiently a category
# generates profit.
#
# Profit Margin =
#
#       Total Profit
#       ------------
#       Total Revenue
#
#       × 100
# ================================================================

query_category_margin = """
SELECT
    Product_Category,

    SUM(Total_Sales) AS Total_Revenue,

    SUM(Profit) AS Total_Profit,

    ROUND(
        SUM(Profit) * 100.0 /
        NULLIF(SUM(Total_Sales), 0),
        2
    ) AS Profit_Margin_Percent

FROM retail_sales

GROUP BY Product_Category

ORDER BY Profit_Margin_Percent DESC;
"""

category_margin_result = run_query(query_category_margin)

print("Profit margin by product category:")
display(category_margin_result)


# ================================================================
# 15. MONTHLY PERFORMANCE WITH RANKING
# ================================================================
#
# Business Question:
# Which months generated the highest revenue?
#
# We first calculate monthly revenue.
#
# Then RANK() assigns a ranking to each month.
#
# This combines:
#     GROUP BY
#     SUM()
#     CTE
#     Window Function
# ================================================================

query_month_rank = """
WITH monthly_performance AS (

    SELECT
        Year,
        Month,
        "Month Name" AS Month_Name,
        SUM(Total_Sales) AS Monthly_Revenue,
        SUM(Profit) AS Monthly_Profit

    FROM retail_sales

    GROUP BY
        Year,
        Month,
        "Month Name"
)

SELECT
    Year,
    Month,
    Month_Name,
    Monthly_Revenue,
    Monthly_Profit,

    RANK() OVER (
        ORDER BY Monthly_Revenue DESC
    ) AS Revenue_Rank

FROM monthly_performance

ORDER BY Revenue_Rank;
"""

month_rank_result = run_query(query_month_rank)

print("Monthly revenue ranking:")
display(month_rank_result)


# ================================================================
# 16. TOP PRODUCTS USING RANKING
# ================================================================
#
# Business Question:
# Which products are the strongest contributors to revenue?
#
# Instead of simply using LIMIT, we use RANK().
#
# This demonstrates a more advanced analytical approach and makes
# the ranking logic explicit.
# ================================================================

query_product_rank = """
WITH product_performance AS (

    SELECT
        Product,
        Product_Category,
        SUM(Total_Sales) AS Total_Revenue,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Units_Sold

    FROM retail_sales

    GROUP BY
        Product,
        Product_Category
)

SELECT
    Product,
    Product_Category,
    Total_Revenue,
    Total_Profit,
    Units_Sold,

    RANK() OVER (
        ORDER BY Total_Revenue DESC
    ) AS Revenue_Rank

FROM product_performance

ORDER BY Revenue_Rank;
"""

product_rank_result = run_query(query_product_rank)

print("Product revenue ranking:")
display(product_rank_result.head(15))


# ================================================================
# 17. FINAL BUSINESS PERFORMANCE TABLE
# ================================================================
#
# This final query combines several analytical techniques into
# one management-oriented result.
#
# It provides:
#
# - Branch
# - Transactions
# - Revenue
# - Profit
# - Profit Margin
# - Revenue Contribution
# - Revenue Rank
#
# This is an example of how SQL can create a dataset suitable for
# reporting or a future Power BI dashboard.
# ================================================================

query_final_branch = """
WITH branch_metrics AS (

    SELECT
        Branch,

        COUNT(DISTINCT Invoice_ID) AS Transactions,

        SUM(Total_Sales) AS Total_Revenue,

        SUM(Profit) AS Total_Profit

    FROM retail_sales

    GROUP BY Branch
)

SELECT
    Branch,
    Transactions,
    Total_Revenue,
    Total_Profit,

    -- Calculate profit margin
    ROUND(
        Total_Profit * 100.0 /
        NULLIF(Total_Revenue, 0),
        2
    ) AS Profit_Margin_Percent,

    -- Calculate each branch's share of total revenue
    ROUND(
        Total_Revenue * 100.0 /
        SUM(Total_Revenue) OVER (),
        2
    ) AS Revenue_Contribution_Percent,

    -- Rank branches by revenue
    RANK() OVER (
        ORDER BY Total_Revenue DESC
    ) AS Revenue_Rank

FROM branch_metrics

ORDER BY Revenue_Rank;
"""

final_branch_result = run_query(query_final_branch)

print("Final branch performance analysis:")
display(final_branch_result)


# ================================================================
# 18. CLOSE THE DATABASE CONNECTION
# ================================================================
#
# We close the SQLite connection after all SQL analysis is complete.
#
# Closing database connections is good programming practice because
# it releases the resources associated with the database.
# ================================================================

conn.close()

print("SQL analysis completed successfully.")
print("SQLite database connection closed.")

Branches generating more than KES 200 million in revenue:


,Branch,Transactions,Total_Revenue,Total_Profit


Product categories generating more than KES 30 million profit:


,Product_Category,Total_Revenue,Total_Profit,Units_Sold
0,Electronics,1.489817e+08,38594506.08,55022
1,Beverages,1.494720e+08,38492597.87,55091
2,Household,1.498403e+08,38468709.21,55756
3,Stationery,1.467696e+08,37931377.66,54661
4,Groceries,1.459827e+08,37873743.28,53994


Example transaction value classifications:


,Invoice_ID,Total_Sales,Transaction_Value_Category
0,INV100000,8035.70,Low Value
1,INV100001,9876.73,Low Value
2,INV100002,21918.34,Medium Value
3,INV100003,19635.20,Medium Value
4,INV100004,18649.80,Medium Value
5,INV100005,25002.13,Medium Value
6,INV100006,17127.79,Medium Value
7,INV100007,4678.95,Low Value
8,INV100008,3370.90,Low Value
9,INV100009,19666.51,Medium Value


... (25 outputs left out)

,Product,Product_Category,Total_Revenue,Total_Profit,Units_Sold,Revenue_Rank
0,Bucket,Household,30678061.78,7843213.27,11032,1
1,Detergent,Household,30487143.99,7789491.97,11605,2
2,Coffee,Beverages,30478338.93,7783734.41,11062,3
3,Power Bank,Electronics,30415971.23,7832803.20,11045,4
4,Soda,Beverages,30404172.09,7770028.94,11092,5
5,Juice,Beverages,30365284.72,7825329.53,11189,6
6,USB Flash,Electronics,30311331.53,7894317.08,11025,7
7,Book,Stationery,29890530.14,7883422.08,10885,8
8,Rice,Groceries,29830726.37,7748124.80,10940,9
9,Headphones,Electronics,29713346.04,7810372.63,11156,10


Final branch performance analysis:


,Branch,Transactions,Total_Revenue,Total_Profit,Profit_Margin_Percent,Revenue_Contribution_Percent,Revenue_Rank
0,Nakuru,7196,1.069901e+08,27530326.44,25.73,14.44,1
1,Mombasa,7272,1.069765e+08,27537270.80,25.74,14.44,2
2,Eldoret,7200,1.064426e+08,27584451.30,25.91,14.36,3
3,Westlands,7015,1.057603e+08,27290291.29,25.80,14.27,4
4,Migori,7012,1.051070e+08,27311467.44,25.98,14.18,5
5,Kisumu,7192,1.050057e+08,27265106.86,25.97,14.17,6
6,Nairobi CBD,7113,1.047641e+08,26842019.97,25.62,14.14,7


SQL analysis completed successfully.
SQLite database connection closed.
